# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [1]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

no stored variable or alias data_set
no stored variable or alias labels
no stored variable or alias test_data_set
no stored variable or alias test_labels
no stored variable or alias unique_labels


## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [2]:
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()

data_set = iris.data[0:len(iris.target)-20,:]
labels = iris.target[0:len(iris.target)-20]
unique_labels = np.unique(iris.target)

test_data_set = iris.data[-20:,:]
test_labels = iris.target[-20:]


import pickle
with open("data_set.pkl", "wb") as f:
    pickle.dump(data_set, f)
with open("labels.pkl", "wb") as f:
    pickle.dump(labels, f)
with open("test_data_set.pkl", "wb") as f:
    pickle.dump(test_data_set, f) 
with open("test_labels.pkl", "wb") as f:
    pickle.dump(test_labels, f)
with open("unique_labels.pkl", "wb") as f:
    pickle.dump(unique_labels, f)


In [3]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [4]:
def build_classifiers():
    from sklearn.linear_model import LogisticRegression
    
    classifiers_list = [
        ('Logistic Regression', LogisticRegression(random_state=42, max_iter=1000)),
        ('KNN', KNeighborsClassifier(n_neighbors=5)),
        ('SVM Linear', SVC(kernel='linear', random_state=42)),
        ('Decision Tree', DecisionTreeClassifier(random_state=42)),
        ('Naive Bayes', GaussianNB()),
        ('QDA', QuadraticDiscriminantAnalysis())
    ]
    
    accuracies = []
    trained_classifiers = []
    
    for name, clf in classifiers_list:
        clf.fit(data_set, labels)
        predictions = clf.predict(test_data_set)
        accuracy = accuracy_score(test_labels, predictions)
        accuracies.append(accuracy)
        trained_classifiers.append((name, clf, accuracy))
        print(f"{name}: {accuracy:.4f}")
    
    trained_classifiers.sort(key=lambda x: x[2], reverse=True)
    best_three = [clf for name, clf, acc in trained_classifiers[:3]]
    
    print(f"\nBest three classifiers:")
    for name, clf, acc in trained_classifiers[:3]:
        print(f"  {name}: {acc:.4f}")
    
    return best_three

In [5]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130,3))
    
    # stacked classifier part:
    from sklearn.linear_model import LogisticRegression
    stacked_classifier = LogisticRegression(random_state=42, max_iter=1000)
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [6]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

Logistic Regression: 0.9000
KNN: 1.0000
SVM Linear: 0.9000
Decision Tree: 0.9000
Naive Bayes: 0.9000
QDA: 0.9500

Best three classifiers:
  KNN: 1.0000
  QDA: 0.9500
  Logistic Regression: 0.9000
0.95


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [7]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_accuracy_vector(predicted, true_labels):
    """Returns binary vector: 1 where prediction is wrong, 0 where correct"""
    return (predicted != true_labels).astype(int)

def calculate_error(model):
    predicted = model.predict(test_set)
    I = calculate_accuracy_vector(predicted, test_labels)
    Z = np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [8]:
def set_new_weights(model):
    predicted = model.predict(test_set)
    I = calculate_accuracy_vector(predicted, test_labels)
    new_weights = (1 + I) / np.sum(1 + I)
    return new_weights

Train the classifier with the code below:

In [9]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00066756 0.00066756 0.00133511 0.00133511 0.00066756 0.00133511
 0.00133511 0.00066756 0.00066756 0.00133511 0.00133511 0.00066756
 0.00133511 0.00066756 0.00133511 0.00133511 0.00133511 0.00133511
 0.00066756 0.00066756 0.00133511 0.00066756 0.00066756 0.00066756
 0.00066756 0.00066756 0.00133511 0.00133511 0.00133511 0.00066756
 0.00133511 0.00066756 0.00133511 0.00066756 0.00133511 0.00133511
 0.00133511 0.00133511 0.00133511 0.00066756 0.00066756 0.00066756
 0.00066756 0.00066756 0.00133511 0.00066756 0.00066756 0.00133511
 0.00133511 0.00133511 0.00066756 0.00133511 0.00133511 0.00133511
 0.00133511 0.00133511 0.00066756 0.00133511 0.00066756 0.00133511
 0.00066756 0.00066756 0.00066756 0.00066756 0.00133511 0.00066756
 0.00066756 0.00066756 0.00133511 0.00133511 0.00066756 0.00133511
 0.00066756 0.00066756 0.00066756 0.00133511 0.00133511 0.00066756
 0.00066756 0.00133511 0.00066756 0.00133511 0.00133511 0.00133511
 0.00066756 0.00133511 0.00133511 0.00133511 0.00133511 0.0006

Set the validation data set:

In [10]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [11]:
def get_prediction(x):
    predictions = []
    for clf in classifiers:
        pred = clf.predict(x)
        predictions.append(pred)
    
    predictions = np.array(predictions)
    
    from scipy import stats
    votes = stats.mode(predictions, axis=0)[0]
    return votes

Test it:

In [12]:
prediction = get_prediction(validate_x)[0]

print(prediction)

1
